# 🎯 音频分类推理脚本（支持 Finetune）

本 Notebook 用于加载训练好的模型进行推理，并支持在测试集的一部分数据上进行 finetune：
- 选择不同的前端（Frontend）类型
- 加载对应的预训练模型权重
- **将 testset 数据分为两部分：一部分用于 finetune，一部分用于测试**
- 对模型进行 finetune（可选）
- 在测试数据集上进行推理（分别使用原始模型和 finetune 后的模型）
- 输出混淆矩阵、分类报告等结果（区分 finetune 和不 finetune 的结果）

---

## 支持的前端类型

| 前端名称 | 描述 | 模型权重 |
|---------|------|----------|
| `fft` | Log-Magnitude FFT 频谱 | `fft_best.weights.h5` |
| `logmel` | Log-Mel 频谱图 | `logmel_best.weights.h5` |
| `pcen` | PCEN (Per-Channel Energy Normalization) | `pcen_best.weights.h5` |
| `mfcc` | MFCC (Mel-Frequency Cepstral Coefficients) | `mfcc_best.weights.h5` |
| `logmel_wiener` | Log-Mel + Wiener 降噪 | `logmel_wiener_best.weights.h5` |
| `pcen_wiener` | PCEN + Wiener 降噪 | `pcen_wiener_real_bias_best.weights.h5` |

## 1. 导入依赖

In [84]:
import os
import sys
import numpy as np
import tensorflow as tf
import librosa
import librosa.display
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.utils import class_weight
import joblib
from IPython.display import display, Audio
import warnings
warnings.filterwarnings('ignore')

# 添加 src 目录到 path
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))
from model import build_model
from keras.optimizers import Adam
from keras.callbacks import ModelCheckpoint, EarlyStopping

# GPU 配置
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print(f"✅ GPU 已就绪: {len(gpus)} 个设备")
else:
    print("⚠️ 未检测到 GPU，使用 CPU 推理")

⚠️ 未检测到 GPU，使用 CPU 推理


## 2. ⚙️ 配置区域

### 2.1 设置测试数据集路径

In [85]:
# =============================================================================
# 🎛️ 测试数据集路径配置
# =============================================================================
# 测试数据集目录 (testset目录包含emergency, movement, unknown三个子目录)
TEST_DATA_DIR = "/Users/zilongzeng/Research/Drone/testset"

# Label Encoder 路径
ENCODER_PATH = "../saved_models/label_encoder.joblib"

# Finetune 配置
FINETUNE_RATIO = 0.3  # 用于 finetune 的数据比例（30%）
FINETUNE_ENABLED = True  # 是否启用 finetune
FINETUNE_EPOCHS = 10  # Finetune 的 epoch 数
FINETUNE_BATCH_SIZE = 32  # Finetune 的 batch size
FINETUNE_LEARNING_RATE = 1e-5  # Finetune 的学习率（通常比初始训练更小）

### 2.2 选择前端类型和模型

In [ ]:
# =============================================================================
# 🎛️ 前端类型和模型路径配置
# =============================================================================
# 选择前端类型（修改这里切换不同的前端）
FRONTEND_TYPE = "pcen_wiener"  # 可选: "fft", "logmel", "pcen", "mfcc", "logmel_wiener", "pcen_wiener"

# 根据前端类型确定模型权重路径
MODEL_WEIGHT_PATHS = {
    "fft": "../saved_models/fft/fft_best.weights.h5",
    "logmel": "../saved_models/logmel/logmel_best.weights.h5",
    "pcen": "../saved_models/pcen/pcen_best.weights.h5",
    "mfcc": "../saved_models/mfcc/mfcc_best.weights.h5",
    "logmel_wiener": "../saved_models/logmel_wiener/logmel_wiener_best.weights.h5",
    "pcen_wiener": "../saved_models/pcen_wiener/pcen_wiener_real_bias_best.weights.h5"
}

MODEL_WEIGHT_PATH = MODEL_WEIGHT_PATHS.get(FRONTEND_TYPE)
if MODEL_WEIGHT_PATH is None:
    raise ValueError(f"未知的前端类型: {FRONTEND_TYPE}")

print(f"📌 前端类型: {FRONTEND_TYPE}")
print(f"📌 模型权重路径: {MODEL_WEIGHT_PATH}")
print(f"📌 Finetune 启用: {FINETUNE_ENABLED}")
if FINETUNE_ENABLED:
    print(f"📌 Finetune 数据比例: {FINETUNE_RATIO:.1%}")
    print(f"📌 Finetune Epochs: {FINETUNE_EPOCHS}")

📌 前端类型: logmel_wiener
📌 模型权重路径: ../saved_models/logmel_wiener/logmel_wiener_best.weights.h5
📌 Finetune 启用: True
📌 Finetune 数据比例: 30.0%
📌 Finetune Epochs: 10


### 2.3 Wiener 降噪配置（如果使用）

In [87]:
# =============================================================================
# 🎛️ Wiener 降噪配置（仅当使用 logmel_wiener 或 pcen_wiener 时需要）
# =============================================================================
ENABLE_WIENER_DENOISE = FRONTEND_TYPE in ["logmel_wiener", "pcen_wiener"]
CALIB_NOISE_WAV = "/Users/zilongzeng/Research/Drone/dataset/raw/tellonoise/19700101_000018.wav"

if ENABLE_WIENER_DENOISE:
    print(f"📌 Wiener 降噪已启用")
    print(f"📌 校准噪声文件: {CALIB_NOISE_WAV}")
else:
    print("📌 Wiener 降噪未启用")

📌 Wiener 降噪已启用
📌 校准噪声文件: /Users/zilongzeng/Research/Drone/dataset/raw/tellonoise/19700101_000018.wav


## 3. 定义前端和特征提取函数

In [88]:
# =============================================================================
# 音频参数
# =============================================================================
SAMPLE_RATE = 16000
DURATION = 1.0
TARGET_LEN = int(DURATION * SAMPLE_RATE)  # 16000

# =============================================================================
# 前端参数
# =============================================================================
N_FFT = 1024
HOP_LENGTH = 512
CENTER = False
N_MELS = 256
N_MFCC = 40
FMIN = 50
FMAX = None
TOP_DB = 80.0
MAX_FRAMES = int(DURATION * SAMPLE_RATE / HOP_LENGTH) + 1  # 32
N_BINS = N_FFT // 2 + 1  # 513 (FFT 前端)

# PCEN 参数
PCEN_KWARGS = dict(gain=0.98, bias=2.0, power=0.5, time_constant=0.06, eps=1e-6)

# =============================================================================
# 根据前端类型确定输入形状
# =============================================================================
def get_input_shape(frontend_type):
    """根据前端类型返回模型输入形状"""
    if frontend_type == "fft":
        return (N_BINS, MAX_FRAMES, 1)
    elif frontend_type == "mfcc":
        return (N_MFCC, MAX_FRAMES, 1)
    else:
        return (N_MELS, MAX_FRAMES, 1)

INPUT_SHAPE = get_input_shape(FRONTEND_TYPE)
print(f"📌 模型输入形状: {INPUT_SHAPE}")

📌 模型输入形状: (256, 32, 1)


In [89]:
# =============================================================================
# Wiener 降噪相关函数
# =============================================================================
NOISE_PROFILE_BASE = None

def build_noise_profile_from_wav(wav_path, seconds=1.0, method="mean"):
    """从校准噪声文件构建噪声功率谱 profile"""
    ns, _ = librosa.load(wav_path, sr=SAMPLE_RATE, mono=True)
    target = int(seconds * SAMPLE_RATE)
    
    if len(ns) < target:
        ns = np.pad(ns, (0, target - len(ns)), mode="wrap")
    else:
        ns = ns[:target]
    
    N = librosa.stft(ns, n_fft=N_FFT, hop_length=HOP_LENGTH, center=CENTER)
    Pn = (np.abs(N) ** 2).astype(np.float32)
    
    if method == "median":
        profile = np.median(Pn, axis=1)
    else:
        profile = np.mean(Pn, axis=1)
    
    return profile.astype(np.float32)

def wiener_denoise_with_profile(y_mix, noise_profile, eps=1e-12):
    """使用固定噪声 profile 的 Wiener 滤波"""
    X = librosa.stft(y_mix, n_fft=N_FFT, hop_length=HOP_LENGTH, center=CENTER)
    Px = (np.abs(X) ** 2).astype(np.float32)
    
    Pn = noise_profile.astype(np.float32)[:, None]
    
    if Pn.shape[0] != Px.shape[0]:
        m = min(Pn.shape[0], Px.shape[0])
        Pn = Pn[:m, :]
        Px = Px[:m, :]
        X = X[:m, :]
    
    Ps = np.maximum(Px - Pn, 0.0)
    G = Ps / (Ps + Pn + eps)
    Y = G * X
    
    y_out = librosa.istft(Y, hop_length=HOP_LENGTH, center=CENTER, length=len(y_mix))
    return y_out.astype(np.float32)

# 如果启用 Wiener，构建噪声 profile
if ENABLE_WIENER_DENOISE and os.path.exists(CALIB_NOISE_WAV):
    NOISE_PROFILE_BASE = build_noise_profile_from_wav(CALIB_NOISE_WAV, seconds=1.0)
    print(f"✅ 噪声 profile 已构建: shape={NOISE_PROFILE_BASE.shape}")
elif ENABLE_WIENER_DENOISE:
    print(f"⚠️ 校准噪声文件不存在: {CALIB_NOISE_WAV}")

⚠️ 校准噪声文件不存在: /Users/zilongzeng/Research/Drone/dataset/raw/tellonoise/19700101_000018.wav


In [90]:
# =============================================================================
# 特征提取函数
# =============================================================================

def extract_fft_features(y):
    """FFT Log-Magnitude 特征"""
    D = librosa.stft(y, n_fft=N_FFT, hop_length=HOP_LENGTH, center=CENTER)
    mag = np.abs(D).astype(np.float32)
    feat = librosa.amplitude_to_db(mag, ref=np.max, top_db=TOP_DB)
    
    if feat.shape[1] < MAX_FRAMES:
        feat = np.pad(feat, ((0, 0), (0, MAX_FRAMES - feat.shape[1])), mode='constant')
    else:
        feat = feat[:, :MAX_FRAMES]
    return feat.astype(np.float32)

def extract_logmel_features(y):
    """Log-Mel 频谱特征"""
    mel = librosa.feature.melspectrogram(
        y=y, sr=SAMPLE_RATE,
        n_fft=N_FFT, hop_length=HOP_LENGTH, center=CENTER,
        n_mels=N_MELS, fmin=FMIN, fmax=FMAX, power=2.0
    )
    feat = librosa.power_to_db(mel, ref=np.max, top_db=TOP_DB)
    
    if feat.shape[1] < MAX_FRAMES:
        feat = np.pad(feat, ((0, 0), (0, MAX_FRAMES - feat.shape[1])), mode='constant')
    else:
        feat = feat[:, :MAX_FRAMES]
    return feat.astype(np.float32)

def extract_pcen_features(y):
    """PCEN 特征"""
    mel = librosa.feature.melspectrogram(
        y=y, sr=SAMPLE_RATE,
        n_fft=N_FFT, hop_length=HOP_LENGTH, center=CENTER,
        n_mels=N_MELS, fmin=FMIN, fmax=FMAX, power=2.0
    )
    feat = librosa.pcen(mel, sr=SAMPLE_RATE, hop_length=HOP_LENGTH, **PCEN_KWARGS)
    
    if feat.shape[1] < MAX_FRAMES:
        feat = np.pad(feat, ((0, 0), (0, MAX_FRAMES - feat.shape[1])), mode='constant')
    else:
        feat = feat[:, :MAX_FRAMES]
    return feat.astype(np.float32)

def extract_mfcc_features(y):
    """MFCC 特征"""
    mfcc = librosa.feature.mfcc(
        y=y, sr=SAMPLE_RATE,
        n_mfcc=N_MFCC, n_mels=N_MELS,
        n_fft=N_FFT, hop_length=HOP_LENGTH, center=CENTER,
        fmin=FMIN, fmax=FMAX
    )
    
    if mfcc.shape[1] < MAX_FRAMES:
        mfcc = np.pad(mfcc, ((0, 0), (0, MAX_FRAMES - mfcc.shape[1])), mode='constant')
    else:
        mfcc = mfcc[:, :MAX_FRAMES]
    return mfcc.astype(np.float32)

def extract_features(y, frontend_type, apply_wiener=False):
    """统一的特征提取接口"""
    # 可选 Wiener 降噪
    if apply_wiener and NOISE_PROFILE_BASE is not None:
        y = wiener_denoise_with_profile(y, NOISE_PROFILE_BASE)
    
    # 根据前端类型提取特征
    if frontend_type == "fft":
        return extract_fft_features(y)
    elif frontend_type == "logmel":
        return extract_logmel_features(y)
    elif frontend_type == "pcen":
        return extract_pcen_features(y)
    elif frontend_type == "mfcc":
        return extract_mfcc_features(y)
    elif frontend_type == "logmel_wiener":
        return extract_logmel_features(y)  # Wiener 已在前面应用
    elif frontend_type == "pcen_wiener":
        return extract_pcen_features(y)    # Wiener 已在前面应用
    else:
        raise ValueError(f"未知的前端类型: {frontend_type}")

def load_audio(filepath, sr=SAMPLE_RATE, duration=DURATION):
    """加载音频文件"""
    y, _ = librosa.load(filepath, sr=sr, mono=True, duration=duration)
    if len(y) < TARGET_LEN:
        y = np.pad(y, (0, TARGET_LEN - len(y)), mode='constant')
    else:
        y = y[:TARGET_LEN]
    return y.astype(np.float32)

print("✅ 特征提取函数已定义")

✅ 特征提取函数已定义


## 4. 加载数据并分割为 Finetune 和 Test 集

In [91]:
# =============================================================================
# 加载标签编码器
# =============================================================================
le = joblib.load(ENCODER_PATH)
class_names = le.classes_
NUM_CLASSES = len(class_names)

print(f"✅ 标签编码器已加载")
print(f"📌 类别数量: {NUM_CLASSES}")
print(f"📌 类别名称: {list(class_names)}")

✅ 标签编码器已加载
📌 类别数量: 3
📌 类别名称: [np.str_('emergency'), np.str_('movement'), np.str_('unknown')]


In [92]:
# =============================================================================
# 从 testset 目录加载数据
# =============================================================================
X_all_paths = []
y_all = []

# 检查testset目录是否存在
if not os.path.isdir(TEST_DATA_DIR):
    raise ValueError(f"测试数据目录不存在: {TEST_DATA_DIR}")

print(f"📂 扫描测试数据目录: {TEST_DATA_DIR}")

# 遍历testset目录下的所有子目录
for item in os.listdir(TEST_DATA_DIR):
    item_path = os.path.join(TEST_DATA_DIR, item)
    
    # 如果是目录，检查是否是已知的类别
    if os.path.isdir(item_path):
        # 检查类别名称是否在标签编码器中
        if item in class_names:
            class_idx = le.transform([item])[0]
            count = 0
            
            # 递归遍历该目录下的所有wav文件
            for root, _, files in os.walk(item_path):
                for f in files:
                    if f.lower().endswith('.wav'):
                        X_all_paths.append(os.path.join(root, f))
                        y_all.append(class_idx)
                        count += 1
            
            print(f"   - {item}: {count} 个样本")
        else:
            print(f"   ⚠️ 跳过未知类别目录: {item}")

X_all_paths = np.array(X_all_paths)
y_all = np.array(y_all)
print(f"✅ 从 {TEST_DATA_DIR} 扫描数据完成")
print(f"📌 总样本数: {len(X_all_paths)}")

# 显示每个类别的样本数
if len(y_all) > 0:
    unique, counts = np.unique(y_all, return_counts=True)
    for cls_idx, cnt in zip(unique, counts):
        print(f"   - {class_names[cls_idx]}: {cnt} 样本")

📂 扫描测试数据目录: /Users/zilongzeng/Research/Drone/testset
   - emergency: 226 个样本
   - movement: 222 个样本
   - unknown: 223 个样本
✅ 从 /Users/zilongzeng/Research/Drone/testset 扫描数据完成
📌 总样本数: 671
   - emergency: 226 样本
   - movement: 222 样本
   - unknown: 223 样本


In [93]:
# =============================================================================
# 将数据分割为 Finetune 集和 Test 集（按类别分层）
# =============================================================================
if FINETUNE_ENABLED:
    # 使用分层采样确保每个类别在 finetune 和 test 集中都有代表性
    X_finetune_paths, X_test_paths, y_finetune, y_test = train_test_split(
        X_all_paths, y_all,
        test_size=1 - FINETUNE_RATIO,
        stratify=y_all,
        random_state=42
    )
    
    print(f"\n📊 数据分割完成:")
    print(f"   - Finetune 集: {len(X_finetune_paths)} 样本 ({FINETUNE_RATIO:.1%})")
    print(f"   - Test 集: {len(X_test_paths)} 样本 ({1-FINETUNE_RATIO:.1%})")
    
    # 显示每个类别在 finetune 集中的分布
    print(f"\n📊 Finetune 集类别分布:")
    unique, counts = np.unique(y_finetune, return_counts=True)
    for cls_idx, cnt in zip(unique, counts):
        print(f"   - {class_names[cls_idx]}: {cnt} 样本")
    
    # 显示每个类别在 test 集中的分布
    print(f"\n📊 Test 集类别分布:")
    unique, counts = np.unique(y_test, return_counts=True)
    for cls_idx, cnt in zip(unique, counts):
        print(f"   - {class_names[cls_idx]}: {cnt} 样本")
else:
    # 不使用 finetune，所有数据都作为测试集
    X_test_paths = X_all_paths
    y_test = y_all
    X_finetune_paths = np.array([])
    y_finetune = np.array([])
    print(f"\n📊 未启用 Finetune，所有数据用于测试:")
    print(f"   - Test 集: {len(X_test_paths)} 样本")


📊 数据分割完成:
   - Finetune 集: 201 样本 (30.0%)
   - Test 集: 470 样本 (70.0%)

📊 Finetune 集类别分布:
   - emergency: 68 样本
   - movement: 66 样本
   - unknown: 67 样本

📊 Test 集类别分布:
   - emergency: 158 样本
   - movement: 156 样本
   - unknown: 156 样本


## 5. 加载模型

In [94]:
# =============================================================================
# 构建并加载模型
# =============================================================================
print(">>> 构建模型...")
print(f"📌 输入形状: {INPUT_SHAPE}")
print(f"📌 输出类别: {NUM_CLASSES}")

# 构建模型
model = build_model(INPUT_SHAPE, NUM_CLASSES)

# 加载预训练权重
if os.path.exists(MODEL_WEIGHT_PATH):
    print(f">>> 加载权重: {MODEL_WEIGHT_PATH}")
    model.load_weights(MODEL_WEIGHT_PATH)
    print("✅ 模型加载成功")
else:
    raise FileNotFoundError(f"模型权重文件不存在: {MODEL_WEIGHT_PATH}")

# 编译模型
model.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print("✅ 模型已编译")

>>> 构建模型...
📌 输入形状: (256, 32, 1)
📌 输出类别: 3
>>> 加载权重: ../saved_models/logmel_wiener/logmel_wiener_best.weights.h5
✅ 模型加载成功
✅ 模型已编译


## 6. Finetune 模型（如果启用）

In [95]:
# =============================================================================
# 数据生成器（用于 Finetune）
# =============================================================================
class FinetuneDataGenerator(tf.keras.utils.Sequence):
    def __init__(self, filepaths, labels, batch_size, num_classes, is_training=True):
        self.filepaths = filepaths
        self.labels = labels
        self.batch_size = batch_size
        self.num_classes = num_classes
        self.is_training = is_training
        self.indexes = np.arange(len(self.filepaths))
        if is_training:
            np.random.shuffle(self.indexes)

    def __len__(self):
        return int(np.ceil(len(self.filepaths) / self.batch_size))

    def __getitem__(self, index):
        batch_indexes = self.indexes[index*self.batch_size:(index+1)*self.batch_size]
        batch_size = len(batch_indexes)
        
        X = np.empty((batch_size, *INPUT_SHAPE), dtype=np.float32)
        y = np.empty(batch_size, dtype=int)

        for i, idx in enumerate(batch_indexes):
            try:
                audio = load_audio(self.filepaths[idx])
            except:
                audio = np.zeros(TARGET_LEN, dtype=np.float32)

            apply_wiener = FRONTEND_TYPE in ["logmel_wiener", "pcen_wiener"]
            feat = extract_features(audio, FRONTEND_TYPE, apply_wiener=apply_wiener)
            X[i] = np.expand_dims(feat, axis=-1)
            y[i] = self.labels[idx]

        return X, tf.keras.utils.to_categorical(y, num_classes=self.num_classes)

print("✅ Finetune 数据生成器已定义")

✅ Finetune 数据生成器已定义


In [96]:
# =============================================================================
# 执行 Finetune
# =============================================================================
model_finetuned = None

if FINETUNE_ENABLED and len(X_finetune_paths) > 0:
    print(f"\n🚀 开始 Finetune 模型...")
    print(f"   - Finetune 样本数: {len(X_finetune_paths)}")
    print(f"   - Epochs: {FINETUNE_EPOCHS}")
    print(f"   - Batch Size: {FINETUNE_BATCH_SIZE}")
    print(f"   - Learning Rate: {FINETUNE_LEARNING_RATE}")
    
    # 创建 finetune 数据生成器
    # 将 finetune 数据进一步分为训练和验证集（用于监控过拟合）
    X_ft_train_paths, X_ft_val_paths, y_ft_train, y_ft_val = train_test_split(
        X_finetune_paths, y_finetune,
        test_size=0.2,
        stratify=y_finetune,
        random_state=42
    )
    
    train_gen = FinetuneDataGenerator(
        X_ft_train_paths, y_ft_train, FINETUNE_BATCH_SIZE, NUM_CLASSES, is_training=True
    )
    val_gen = FinetuneDataGenerator(
        X_ft_val_paths, y_ft_val, FINETUNE_BATCH_SIZE, NUM_CLASSES, is_training=False
    )
    
    # 计算类别权重（用于处理类别不平衡）
    class_weights = class_weight.compute_class_weight(
        'balanced', classes=np.unique(y_ft_train), y=y_ft_train
    )
    class_weight_dict = dict(enumerate(class_weights))
    
    # 创建 finetune 后的模型（复制原始模型）
    model_finetuned = build_model(INPUT_SHAPE, NUM_CLASSES)
    model_finetuned.load_weights(MODEL_WEIGHT_PATH)
    
    # 使用较小的学习率进行 finetune
    model_finetuned.compile(
        optimizer=Adam(learning_rate=FINETUNE_LEARNING_RATE),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    
    # 设置回调函数
    finetune_ckpt_path = f"../saved_models/{FRONTEND_TYPE}/finetuned_best.weights.h5"
    os.makedirs(os.path.dirname(finetune_ckpt_path), exist_ok=True)
    
    callbacks = [
        ModelCheckpoint(
            finetune_ckpt_path,
            save_best_only=True,
            monitor='val_accuracy',
            save_weights_only=True,
            verbose=1
        ),
        EarlyStopping(
            patience=5,
            restore_best_weights=True,
            monitor='val_accuracy',
            verbose=1
        )
    ]
    
    # 执行 finetune
    history = model_finetuned.fit(
        train_gen,
        validation_data=val_gen,
        epochs=FINETUNE_EPOCHS,
        callbacks=callbacks,
        class_weight=class_weight_dict,
        verbose=1
    )
    
    # 加载最佳权重
    model_finetuned.load_weights(finetune_ckpt_path)
    
    print(f"\n✅ Finetune 完成！")
    print(f"   - 最佳验证准确率: {max(history.history['val_accuracy']):.4f}")
else:
    print("\n📌 Finetune 未启用或没有 finetune 数据，跳过 finetune 步骤")


🚀 开始 Finetune 模型...
   - Finetune 样本数: 201
   - Epochs: 10
   - Batch Size: 32
   - Learning Rate: 1e-05
Epoch 1/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 172ms/step - accuracy: 0.6594 - loss: 0.7196
Epoch 1: val_accuracy improved from None to 0.65854, saving model to ../saved_models/logmel_wiener/finetuned_best.weights.h5
5/5 ━━━━━━━━━━━━━━━━━━━━ 4s 281ms/step - accuracy: 0.6562 - loss: 0.7361 - val_accuracy: 0.6585 - val_loss: 0.6237
Epoch 2/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 169ms/step - accuracy: 0.7069 - loss: 0.6605
Epoch 2: val_accuracy improved from 0.65854 to 0.68293, saving model to ../saved_models/logmel_wiener/finetuned_best.weights.h5
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 207ms/step - accuracy: 0.7063 - loss: 0.6337 - val_accuracy: 0.6829 - val_loss: 0.5879
Epoch 3/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 176ms/step - accuracy: 0.7371 - loss: 0.6190
Epoch 3: val_accuracy improved from 0.68293 to 0.73171, saving model to ../saved_models/logmel_wiener/finetuned_best.weights.h5
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 215

## 7. 在测试集上进行推理

In [97]:
# =============================================================================
# 准备测试数据
# =============================================================================
print(f"\n📊 准备测试数据...")
print(f"   - 测试样本数: {len(X_test_paths)}")

# 提取所有测试样本的特征
X_test_features = []
for i, path in enumerate(X_test_paths):
    if (i + 1) % 100 == 0:
        print(f"   处理进度: {i+1}/{len(X_test_paths)}")
    
    try:
        audio = load_audio(path)
        apply_wiener = FRONTEND_TYPE in ["logmel_wiener", "pcen_wiener"]
        feat = extract_features(audio, FRONTEND_TYPE, apply_wiener=apply_wiener)
        X_test_features.append(feat)
    except Exception as e:
        print(f"   ⚠️ 处理文件失败 {path}: {e}")
        # 使用零特征作为占位符
        feat_shape = INPUT_SHAPE[:2]  # (height, width)
        X_test_features.append(np.zeros(feat_shape, dtype=np.float32))

X_test_features = np.array(X_test_features)
X_test_features = np.expand_dims(X_test_features, axis=-1)  # 添加 channel 维度

print(f"✅ 测试特征提取完成")
print(f"   - 特征形状: {X_test_features.shape}")


📊 准备测试数据...
   - 测试样本数: 470
   处理进度: 100/470
   处理进度: 200/470
   处理进度: 300/470
   处理进度: 400/470
✅ 测试特征提取完成
   - 特征形状: (470, 256, 32, 1)


In [98]:
# =============================================================================
# 使用原始模型进行推理
# =============================================================================
print(f"\n🔍 使用原始模型进行推理...")
y_pred_original = np.argmax(model.predict(X_test_features, verbose=0), axis=1)
y_pred_proba_original = model.predict(X_test_features, verbose=0)

accuracy_original = accuracy_score(y_test, y_pred_original)
print(f"✅ 原始模型推理完成")
print(f"   - 准确率: {accuracy_original:.4f}")


🔍 使用原始模型进行推理...
✅ 原始模型推理完成
   - 准确率: 0.6574


In [99]:
# =============================================================================
# 使用 Finetune 后的模型进行推理（如果可用）
# =============================================================================
y_pred_finetuned = None
y_pred_proba_finetuned = None
accuracy_finetuned = None

if model_finetuned is not None:
    print(f"\n🔍 使用 Finetune 后的模型进行推理...")
    y_pred_finetuned = np.argmax(model_finetuned.predict(X_test_features, verbose=0), axis=1)
    y_pred_proba_finetuned = model_finetuned.predict(X_test_features, verbose=0)
    
    accuracy_finetuned = accuracy_score(y_test, y_pred_finetuned)
    print(f"✅ Finetune 模型推理完成")
    print(f"   - 准确率: {accuracy_finetuned:.4f}")
    print(f"   - 准确率提升: {accuracy_finetuned - accuracy_original:+.4f}")
else:
    print("\n📌 未进行 Finetune，跳过 Finetune 模型推理")


🔍 使用 Finetune 后的模型进行推理...
✅ Finetune 模型推理完成
   - 准确率: 0.7277
   - 准确率提升: +0.0702


## 8. 生成评估报告和可视化

In [100]:
# =============================================================================
# 保存结果配置
# =============================================================================
SAVE_RESULTS = True
BASE_OUTPUT_DIR = "../result"

# 创建输出目录（区分 finetune 和不 finetune）
if FINETUNE_ENABLED and model_finetuned is not None:
    OUTPUT_DIR_ORIGINAL = os.path.join(BASE_OUTPUT_DIR, f"inference_{FRONTEND_TYPE}_original")
    OUTPUT_DIR_FINETUNED = os.path.join(BASE_OUTPUT_DIR, f"inference_{FRONTEND_TYPE}_finetuned")
    os.makedirs(OUTPUT_DIR_ORIGINAL, exist_ok=True)
    os.makedirs(OUTPUT_DIR_FINETUNED, exist_ok=True)
else:
    OUTPUT_DIR_ORIGINAL = os.path.join(BASE_OUTPUT_DIR, f"inference_{FRONTEND_TYPE}")
    OUTPUT_DIR_FINETUNED = None
    os.makedirs(OUTPUT_DIR_ORIGINAL, exist_ok=True)

print(f"📌 结果保存目录:")
print(f"   - 原始模型: {OUTPUT_DIR_ORIGINAL}")
if OUTPUT_DIR_FINETUNED:
    print(f"   - Finetune 模型: {OUTPUT_DIR_FINETUNED}")

📌 结果保存目录:
   - 原始模型: ../result/inference_logmel_wiener_original
   - Finetune 模型: ../result/inference_logmel_wiener_finetuned


In [101]:
# =============================================================================
# 生成并保存原始模型的评估报告
# =============================================================================
def save_evaluation_results(y_true, y_pred, y_proba, output_dir, model_name="原始模型"):
    """保存评估结果（分类报告、混淆矩阵、预测结果）"""
    
    # 分类报告
    report = classification_report(
        y_true, y_pred,
        target_names=[str(name) for name in class_names],
        output_dict=True
    )
    
    # 保存文本报告
    report_path = os.path.join(output_dir, "classification_report.txt")
    with open(report_path, 'w', encoding='utf-8') as f:
        f.write(f"{model_name} - 分类报告\n")
        f.write("=" * 60 + "\n\n")
        f.write(classification_report(
            y_true, y_pred,
            target_names=[str(name) for name in class_names]
        ))
        f.write(f"\n\n总体准确率: {accuracy_score(y_true, y_pred):.4f}\n")
    
    # 混淆矩阵
    cm = confusion_matrix(y_true, y_pred)
    
    plt.figure(figsize=(10, 8))
    sns.heatmap(
        cm, annot=True, fmt='d', cmap='Blues',
        xticklabels=[str(name) for name in class_names],
        yticklabels=[str(name) for name in class_names]
    )
    plt.title(f'{model_name} - 混淆矩阵')
    plt.ylabel('真实标签')
    plt.xlabel('预测标签')
    plt.tight_layout()
    
    cm_path = os.path.join(output_dir, "confusion_matrix.png")
    plt.savefig(cm_path, dpi=150, bbox_inches='tight')
    plt.close()
    
    # 保存预测结果
    np.savez(
        os.path.join(output_dir, "predictions.npz"),
        y_true=y_test,
        y_pred=y_pred,
        y_proba=y_proba,
        filepaths=X_test_paths
    )
    
    print(f"✅ {model_name} 结果已保存到: {output_dir}")
    print(f"   - classification_report.txt")
    print(f"   - confusion_matrix.png")
    print(f"   - predictions.npz")

if SAVE_RESULTS:
    print(f"\n💾 保存原始模型评估结果...")
    save_evaluation_results(
        y_test, y_pred_original, y_pred_proba_original,
        OUTPUT_DIR_ORIGINAL, model_name="原始模型"
    )


💾 保存原始模型评估结果...
✅ 原始模型 结果已保存到: ../result/inference_logmel_wiener_original
   - classification_report.txt
   - confusion_matrix.png
   - predictions.npz


In [102]:
# =============================================================================
# 生成并保存 Finetune 模型的评估报告（如果可用）
# =============================================================================
if SAVE_RESULTS and model_finetuned is not None and OUTPUT_DIR_FINETUNED:
    print(f"\n💾 保存 Finetune 模型评估结果...")
    save_evaluation_results(
        y_test, y_pred_finetuned, y_pred_proba_finetuned,
        OUTPUT_DIR_FINETUNED, model_name="Finetune 模型"
    )


💾 保存 Finetune 模型评估结果...
✅ Finetune 模型 结果已保存到: ../result/inference_logmel_wiener_finetuned
   - classification_report.txt
   - confusion_matrix.png
   - predictions.npz


In [103]:
# =============================================================================
# 对比结果总结
# =============================================================================
print(f"\n" + "=" * 60)
print(f"📊 推理结果总结")
print(f"=" * 60)
print(f"\n📌 测试集信息:")
print(f"   - 测试样本数: {len(X_test_paths)}")
unique, counts = np.unique(y_test, return_counts=True)
for cls_idx, cnt in zip(unique, counts):
    print(f"   - {class_names[cls_idx]}: {cnt} 样本")

print(f"\n📌 原始模型结果:")
print(f"   - 准确率: {accuracy_original:.4f}")

if model_finetuned is not None:
    print(f"\n📌 Finetune 模型结果:")
    print(f"   - 准确率: {accuracy_finetuned:.4f}")
    print(f"   - 准确率变化: {accuracy_finetuned - accuracy_original:+.4f} ({((accuracy_finetuned - accuracy_original) / accuracy_original * 100):+.2f}%)")
    
    if FINETUNE_ENABLED:
        print(f"\n📌 Finetune 信息:")
        print(f"   - Finetune 样本数: {len(X_finetune_paths)}")
        print(f"   - Finetune Epochs: {FINETUNE_EPOCHS}")
        print(f"   - Finetune 学习率: {FINETUNE_LEARNING_RATE}")

print(f"\n" + "=" * 60)


📊 推理结果总结

📌 测试集信息:
   - 测试样本数: 470
   - emergency: 158 样本
   - movement: 156 样本
   - unknown: 156 样本

📌 原始模型结果:
   - 准确率: 0.6574

📌 Finetune 模型结果:
   - 准确率: 0.7277
   - 准确率变化: +0.0702 (+10.68%)

📌 Finetune 信息:
   - Finetune 样本数: 201
   - Finetune Epochs: 10
   - Finetune 学习率: 1e-05



---

## 📌 使用说明

1. **选择前端类型**: 在 Cell 2.2 中修改 `FRONTEND_TYPE` 变量
2. **配置 Finetune**: 在 Cell 2.1 中设置 `FINETUNE_ENABLED`, `FINETUNE_RATIO`, `FINETUNE_EPOCHS` 等参数
3. **指定测试数据**: 在 Cell 2.1 中设置 `TEST_DATA_DIR` (默认: `/Users/zilongzeng/Research/Drone/testset`)
4. **Wiener 配置**: 如果使用 `logmel_wiener` 或 `pcen_wiener`，确保 Cell 2.3 中的噪声文件路径正确
5. **运行所有 Cell**: 依次运行所有 Cell 进行数据加载、finetune（如果启用）和推理
6. **查看结果**: 结果会保存在 `result/inference_{FRONTEND_TYPE}_original/` 和 `result/inference_{FRONTEND_TYPE}_finetuned/` 目录中（如果启用 finetune）

---